In [1]:
# LOAD
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import faiss
import numpy as np


chunks = np.load("Data/chunks.npy", allow_pickle=True)
index = faiss.read_index("Data/index.faiss")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

c:\Users\phili\Personal\University\Licenta\Proiect\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6136.67it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6680.75it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when lo

In [2]:
# RETRIEVAL 
# retrieve top 20 chunks with the embedding model -> choose top 5 with the reranker model

def retrieve(query, k_initial=20, k_final=5):
    # Pasul A: Retrieval inițial (extrage mai multe fragmente)
    query_prefixed = "query: " + query
    q_emb = embedding_model.encode([query_prefixed])
    distances, indices = index.search(np.array(q_emb), k_initial)
    
    initial_chunks = [chunks[i] for i in indices[0]]
    
    # Pasul B: Reranking (Cross-Encoding)
    # Modelul compară perechi (întrebare, fragment) și dă un scor de relevanță
    sentence_combinations = [[query, chunk] for chunk in initial_chunks]
    rerank_scores = reranker_model.predict(sentence_combinations)
    
    # Pasul C: Sortăm fragmentele în funcție de scorul Reranker-ului
    reranked_results = sorted(
        list(zip(initial_chunks, rerank_scores)), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    # Returnăm doar primele k_final cele mai bune
    return [res[0] for res in reranked_results[:k_final]]

QUESTION ANSWERING LLM

In [3]:
# QUESTION ANSWERING LLM
import requests

def ask_llm(question, context_chunks):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
Ești un asistent virtual oficial care ajută candidații cu informații despre admiterea la FMI UBB Cluj.
Trebuie să fii clar, concis și să răspunzi DOAR pe baza contextului oferit.

REGULI STRICTE:
1. Răspunde EXCLUSIV în limba română.
2. Folosește DOAR informațiile din secțiunea "Context". Nu folosi cunoștințele tale generale.
3. Dacă răspunsul nu se află în "Context", nu încerca să ghicești. Scrie EXACT următoarea propoziție: "Nu dețin această informație conform datelor extrase de pe site-ul oficial."
4. Nu inventa formule de calcul, date sau taxe.

Context:
{context}

Întrebare:
{question}

Răspuns:
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-4-e4b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

SCORING LLM

In [ ]:
# SCORING SYSTEM LLM
def score_llm(question, context_chunks, answer, gold_answer):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
You are a ruthless and objective evaluator of a RAG (Retrieval-Augmented Generation) chatbot for university admissions.
Your task is to evaluate the chatbot's answer. The chatbot answered on a context it has been given.
You will receive the question, the chatbot's answer for it, the context based on which the chatbot has answered, and a gold answer.
Use the reference context to determine whether the chatbot hallucinated or not.
Use the gold answer to determine the factual correctness and the completeness of the chatbot's answer.

When the Golden Answer contains facts absent from the Reference Context, this
indicates a RETRIEVAL failure, not a hallucination — score no_hallucination based
strictly on consistency with the context, and score factual/completeness based on
the Golden Answer.

Question:
{question}

Chatbot Answer:
{answer}

Reference Context:
{context}

Gold Answer:
{gold_answer}

---
SCORING RUBRIC (Be extremely strict!):
1. Factual correctness (0-10): 
   - 10 = Perfectly aligned with the golden answer.
   - 5 = Partially correct, but contains errors.
   - 0 = The information is completely wrong or invented.
2. Completeness (0-10): Did the answer cover all the key facts present in the Golden Answer?
   - 10 = All key facts from the gold answer are present.
   - 5 = Some key facts missing.
   - 0 = Most key facts missing.
3. Clarity (0-10): Is the answer easy to read and in Romanian? (Score 0 if the answer is in English!).
4. Hallucinations penalty (0-10): 
   - 10 = NO hallucinations at all. Only used the context.
   - 0 = The model invented rules, formulas, or deadlines NOT present in the Reference Context.

Return ONLY valid JSON format:
{{
  "factual": <int>,
  "complete": <int>,
  "clarity": <int>,
  "no_hallucination": <int>,
  "brief_reason": "<1-2 sentences explaining exactly why points were deducted>"
}}
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-2-9b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

In [5]:
# QA Model function
def answer_question(question, relevant_chunks):
    answer = ask_llm(question, relevant_chunks)
    return answer

# LLM-as-a-Judge Model function
def score_answer(question, relevant_chunks, answer):
    answer = score_llm(question, relevant_chunks, answer)
    return answer

Model answers question & Judge scores them

In [24]:
import time
import json
import pandas as pd


def load_questions():
    with open("../2nd_QAAndJudgeManualTests/Data/questions.json", "r", encoding="utf-8") as f:
        return json.load(f)

def load_gold_answers():
    pass


def save_results(experimentType, path, answers, scores):
    df = pd.DataFrame(answers)
    df.to_csv(experimentType + "/" + path + "/answers.csv", index=False, encoding="utf-8")

    df = pd.DataFrame(scores)
    df.to_csv(experimentType + "/" + path + "/scores.csv", index=False, encoding="utf-8")


# Compute the total score of the judge given scores
def compute_score(raw: str) -> dict:
    # Each metric has its own value
    WEIGHTS = {
        "factual": 0.40,
        "complete": 0.25,
        "clarity": 0.15,
        "no_hallucination": 0.20,
    }

    # extract the json
    raw = raw.strip()
    start, end = raw.find("{"), raw.rfind("}")
    if start != -1 and end > start:
        data = json.loads(raw[start:end + 1])
    else:
        raise ValueError(f"Nu am putut extrage JSON din răspunsul judge-ului:\n{raw}")

    components = {}
    for key in WEIGHTS:
        val = float(data.get(key, 0))          # coerce, în caz că vine "10" sau lipsește
        components[key] = max(0.0, min(10.0, val))  # clamp în [0, 10]

    final = sum(WEIGHTS[k] * components[k] for k in WEIGHTS)
    data["final_score"] = round(final, 2)
    return data



def answerAndScore(path=None, initialK=20, finalK=5, question=None, experimentType="Manual Experiments"):
    questions = load_questions()
    answers = []
    scores = []
    totalTime = 0
    
    ct = 1
    if path != None and question == None:
        for q in questions[0:1]:
            start = time.time()
            relevant_chunks = retrieve(q["question"], initialK, finalK)

            answer = answer_question(q["question"], relevant_chunks)
            print("Answered question ", ct) 
            answers.append({
                "id": q["id"],
                "question": q["question"],
                "category": q["category"],
                "difficulty": q["difficulty"],
                "answer": answer
            })

            raw = score_answer(q, relevant_chunks, answer)
            try:
                score = compute_score(raw)
            except (ValueError, KeyError, json.JSONDecodeError) as e:
                score = {"error": str(e), "raw": raw, "final_score": None}

            print("Scored answer ", ct) ; ct+=1
            scores.append({
                "id": q["id"],
                "question": q["question"],
                "category": q["category"],
                "difficulty": q["difficulty"],
                "answer": answer,
                "score": json.dumps(score, ensure_ascii=False)
            })

            elapsed = int(time.time() - start) ; minutes = elapsed // 60 ; seconds = elapsed % 60
            print(f"Time: {minutes}m{seconds}s")
            totalTime += elapsed

            save_results(experimentType, path, answers, scores)
            
        print(f"Total time: {totalTime//60}m{totalTime%60}s")


    elif path == None and question != None:
        relevant_chunks = retrieve(question, initialK, finalK)
        answer = answer_question(question, relevant_chunks)
        print(answer) 
        score = score_answer(question, relevant_chunks, answer)
        print("\n\n",score) ; ct+=1
    else:
        print("Wrong parameter input")

Analyse scores

In [33]:
import pandas as pd
import json


def parse_score(text):
    text = str(text).strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    return json.loads(text)


def analyse_scores(path, file="scores.csv", experimentType="Manual Experiments"):
    df = pd.read_csv(experimentType + "/" + path + "/" + file)

    df["score_dict"] = df["score"].apply(parse_score)

    df["final_score"] = df["score_dict"].apply(lambda x: x.get("final_score", 0))
    df["factual"] = df["score_dict"].apply(lambda x: x.get("factual", 0))
    df["complete"] = df["score_dict"].apply(lambda x: x.get("complete", 0))
    df["clarity"] = df["score_dict"].apply(lambda x: x.get("clarity", 0))
    df["no_hallucination"] = df["score_dict"].apply(lambda x: x.get("no_hallucination", 0))

    print("Average final:", df["final_score"].mean())
    print("Average factual:", df["factual"].mean())
    print("Average complete:", df["complete"].mean())
    print("Average clarity:", df["clarity"].mean())
    print("Average non-hallucinative:", df["no_hallucination"].mean())
    print("-------------------------------------")
    print(df.groupby("difficulty")["final_score"].mean())
    print(df.groupby("category")["final_score"].mean())

Manual Experiments

In [6]:
#answerAndScore("1_newInput")
# Total time: 5m9s

analyse_scores("1_newInput")

Average final: 8.73
Average factual: 8.65

difficulty
easy      8.987500
medium    8.966667
tricky    8.150000
Name: final_score, dtype: float64
category
calendar_admitere      7.866667
cazuri_edge_case       8.325000
contestatii           10.000000
documente_necesare    10.000000
inscriere_online       9.500000
locuri_buget           9.500000
medii_minime           8.400000
probe                  9.000000
specializari           8.525000
taxe                   9.300000
Name: final_score, dtype: float64


In [18]:
#answerAndScore("2_changedModels")
# Total time: 

# Improvement is very clear: the QA model answers much more reliably.

analyse_scores("2_changedModels")

Average final: 8.725
Average factual: 9.65

difficulty
easy      8.718750
medium    8.541667
tricky    8.916667
Name: final_score, dtype: float64
category
calendar_admitere      8.666667
cazuri_edge_case       9.000000
contestatii            9.250000
documente_necesare    10.000000
inscriere_online      10.000000
locuri_buget          10.000000
medii_minime           6.750000
probe                 10.000000
specializari           7.437500
taxe                  10.000000
Name: final_score, dtype: float64


In [34]:
#answerAndScore("3_computeScore")

analyse_scores("3_computeScore")

Average final: 10.0
Average factual: 10.0
Average complete: 10.0
Average clarity: 10.0
Average non-hallucinative: 10.0
-------------------------------------
difficulty
easy    10.0
Name: final_score, dtype: float64
category
documente_necesare    10.0
Name: final_score, dtype: float64
